In [1]:
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template
import pandas as pd
import torch
from datasets import Dataset
from trl import SFTTrainer, SFTConfig

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


d:\Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0325 05:28:05.107000 4892 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [2]:
def merge_tsv(file1, file2, output_name):
    df1 = pd.read_csv(file1, sep='\t')
    df2 = pd.read_csv(file2, sep='\t')
    
    combined = pd.concat([df1, df2], ignore_index=True)
    combined.to_csv(output_name, sep='\t', index=False)

dir= "../dataset/"
merge_tsv(dir + 'train1/prompts_train1.tsv', dir + 'train2/prompts.tsv', dir + 'conc_prompts.tsv')

# Пара 2
merge_tsv(dir + 'train1.tsv', dir + 'train2.tsv', dir + 'conc_train.tsv')

In [3]:
datap = "../dataset/conc_prompts.tsv"
datat = "../dataset/conc_train.tsv"

df_data = pd.read_csv(datat, sep='\t')
df_prompts = pd.read_csv(datap, sep='\t')

df_merged = pd.merge(df_data, df_prompts, on='id', how='inner')

system_prompt = """You are a scientific visualization expert and prompt engineer.

Create a prompt for an AI image generator that will produce an accurate and visually compelling cover image for a scientific paper.

CRITICAL RULES:
- Promts size should be 50-150 tokens
- Scientific accuracy: Visual elements must correctly represent the scientific concept
- No fictional or decorative elements that contradict the research
- Use correct scientific terminology in the prompt
- Output ONLY the prompt, no extra text
- Image must be in A4 format (portrait orientation, 1:sqrt(2) aspect ratio). Add this information to prompt.

PROMPT COMPONENTS:
1. Core scientific concept (what is being shown)
2. Specific visual details (colors, materials, lighting)
3. Style appropriate for the journal/conference
4. Composition and perspective

EXAMPLES:

Article: "Graphene-Based Battery with 10x Capacity"
Prompt: "Cross-section of graphene layered anode material, lithium ions intercalating between graphene sheets, electron flow visualized as golden energy streams, blue and gray color palette, scientific illustration style, cutaway view, detailed material texture"

Article: "Neural Network Explains Visual Cortex Activity"
Prompt: "Artificial neural network architecture overlaid on primate visual cortex diagram, activation patterns shown as colored heatmaps, connections between nodes mimicking biological pathways, academic illustration style, split-view showing both AI and biology, cool blue to warm red gradient
"""


formatted_data = []

for index, row in df_merged.iterrows():
    title = str(row['title']) if pd.notna(row['title']) else ""
    abstract = str(row['abstract']) if pd.notna(row['abstract']) else ""
    abstract_text = (title + " " + abstract).strip()
    
    prompts = ['prompt#0', 'prompt#1'] 
    
    for p_col in prompts:
        teacher_answer = str(row[p_col])
        if teacher_answer.strip() and teacher_answer != 'nan':
            dialogue = {
                "messages": [
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": f"Abstract:\n{abstract_text}"},
                    {"role": "assistant", "content": teacher_answer}
                ]
            }
            formatted_data.append(dialogue)

df_formatted = pd.DataFrame(formatted_data)
dataset = Dataset.from_pandas(df_formatted)
print(f"Всего примеров {len(dataset)}")

Всего примеров 18472


In [4]:
max_seq_length = 1024 #удваиваем токеты, чтобы запустить сразу несколько тренировок
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Meta-Llama-3.1-8B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

#LoRA
model = FastLanguageModel.get_peft_model(
    model,
    r = 8,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 8,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3.1",
)

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]
    return { "text" : texts}

#Сюда датасет надо
dataset = dataset.map(formatting_prompts_func, batched = True)

print("можно обучать")

==((====))==  Unsloth 2026.3.10: Fast Llama patching. Transformers: 5.3.0.
   \\   /|    NVIDIA GeForce RTX 4060. Num GPUs = 1. Max memory: 7.995 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.10.0+cu126. CUDA: 8.9. CUDA Toolkit: 12.6. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights: 100%|██████████| 291/291 [00:02<00:00, 113.76it/s]
Unsloth: Will load unsloth/meta-llama-3.1-8b-instruct-unsloth-bnb-4bit as a legacy tokenizer.
Unsloth 2026.3.10 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.
Map: 100%|██████████| 18472/18472 [00:01<00:00, 15611.01 examples/s]

можно обучать


In [5]:
from transformers import TrainerCallback
import torch
import gc

class ClearMemoryCallback(TrainerCallback):
    def on_step_end(self, args, state, control, **kwargs):
        if state.global_step % 20 == 0:
            gc.collect()
            torch.cuda.empty_cache()

In [6]:
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    callbacks = [ClearMemoryCallback()],
    args = SFTConfig(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 16,
        warmup_steps = 0.1,
        num_train_epochs = 1,
        learning_rate = 1e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 10,
        optim = "paged_adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = 3407,
        output_dir = "outputs",
        save_strategy = "steps",
        save_steps = 100,
        report_to = "none",
    ),
)

trainer.train(resume_from_checkpoint="outputs/checkpoint-300")

Unsloth: Tokenizing ["text"]: 100%|██████████| 18472/18472 [00:25<00:00, 715.64 examples/s]


🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 18,472 | Num Epochs = 1 | Total steps = 1,155
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 16
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 16 x 1) = 16
 "-____-"     Trainable parameters = 20,971,520 of 8,051,232,768 (0.26% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
310,0.941797
320,0.927563
330,0.947046
340,0.938479
350,0.952925
360,0.934405
370,0.952465
380,0.938674
390,0.942717
400,0.933560


TrainOutput(global_step=1155, training_loss=0.6804931562184255, metrics={'train_runtime': 38485.6183, 'train_samples_per_second': 0.48, 'train_steps_per_second': 0.03, 'total_flos': 5.6382457915131494e+17, 'train_loss': 0.6804931562184255, 'epoch': 1.0})

In [ ]:
model.save_pretrained("lora_backup")
tokenizer.save_pretrained("lora_backup")

In [1]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "lora_backup",
    max_seq_length = 1024,
    dtype = None,
    load_in_4bit = True,
)

print("Модель успешно загружена! Начинаем конвертацию в GGUF...")

# Запускаем упаковку
model.save_pretrained_gguf(
    "llama_for_article_cover_promt_GGUF",
    tokenizer,
    quantization_method = "q4_k_m"
)

print("Готово! Ваш GGUF файл ждет вас в папке проекта.")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


d:\Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0325 19:51:52.398000 15020 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.10: Fast Llama patching. Transformers: 5.3.0.
   \\   /|    NVIDIA GeForce RTX 4060. Num GPUs = 1. Max memory: 7.995 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.10.0+cu126. CUDA: 8.9. CUDA Toolkit: 12.6. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights: 100%|██████████| 291/291 [00:07<00:00, 40.23it/s]
Unsloth: Will load unsloth/meta-llama-3.1-8b-instruct-unsloth-bnb-4bit as a legacy tokenizer.
Unsloth 2026.3.10 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


Модель успешно загружена! Начинаем конвертацию в GGUF...
Unsloth: Merging model weights to 16-bit format...
Found HuggingFace hub cache directory: C:\Users\Burgom\.cache\huggingface\hub


...

Unsloth: Preparing safetensor model files: 100%|██████████| 4/4 [00:00<00:00, 4425.54it/s]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [00:39<00:00,  9.84s/it]


Unsloth: Merge process complete. Saved to `d:\Project\ArticleCover\src\llama_for_article_cover_promt_GGUF`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF bf16 might take 3 minutes.
\        /    [2] Converting GGUF bf16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Missing packages: build-essential openssl
Unsloth: Will attempt to install missing system packages.


RuntimeError: Unsloth: GGUF conversion failed: Unsloth: Missing 'build-essential' and winget not available.
Install manually: winget install Microsoft.VisualStudio.2022.BuildTools